In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import json, os, re, time, subprocess, ast
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

INPUT_DIR = '/kaggle/input/datasets/apr-dataset'
OUTPUT_DIR = '/kaggle/working'

KB_PATH = f'{INPUT_DIR}/pattern_knowledge_base_final.json'
TEST_PATH = f'{INPUT_DIR}/test_loop_filtered_verified.jsonl'
TEST_CASES_DIR = f'{INPUT_DIR}/public_test_cases/codenet/public_test_cases'
BASELINE_OUTPUT_PATH = f'{OUTPUT_DIR}/phase2_baseline_singleLLM_gpt54mini.jsonl'

SPEEDUP_THRESHOLD = 1.10
MAX_ATTEMPTS = 7
GPT_MODEL = "gpt-5.4-mini"

user_secrets = UserSecretsClient()
client = OpenAI(api_key=user_secrets.get_secret("OPENAI_API_KEY"))

with open(KB_PATH) as f:
    pattern_knowledge_base = json.load(f)
print(f'Loaded KB: {len(pattern_knowledge_base)} patterns')

test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

print('Test cases dir exists:', os.path.isdir(TEST_CASES_DIR))

In [ ]:
def call_gen_llm(prompt, max_new_tokens=800, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.15,
                max_completion_tokens=max_new_tokens,
            )
            content = response.choices[0].message.content
            return content.strip() if content else ''
        except Exception as e:
            err_str = str(e).lower()
            if 'rate_limit' in err_str or '429' in err_str:
                wait_time = 15 * (attempt + 1)
                print(f'   Rate limited, waiting {wait_time}s...')
                time.sleep(wait_time)
            else:
                print(f'   API error: {e}')
                return ''
    return ''

print(f'{GPT_MODEL} ready.')

In [ ]:
def extract_code_from_response(response):
    match = re.search(r"```(?:python)?\s*\n(.*?)```", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

print('Code extractor ready.')

In [ ]:
def load_test_cases(problem_id, test_cases_dir):
    problem_dir = os.path.join(test_cases_dir, problem_id)
    if not os.path.isdir(problem_dir):
        return []
    cases = []
    i = 0
    while True:
        input_path = os.path.join(problem_dir, f'input.{i}.txt')
        output_path = os.path.join(problem_dir, f'output.{i}.txt')
        if not os.path.exists(input_path):
            break
        with open(input_path) as f:
            stdin_data = f.read()
        with open(output_path) as f:
            expected_output = f.read().strip()
        cases.append({'stdin': stdin_data, 'expected': expected_output})
        i += 1
    return cases

def check_compiles(code):
    try:
        compile(code, '<repair>', 'exec')
        return True, None
    except SyntaxError as e:
        return False, str(e)

def run_with_timeout(code, stdin_data, timeout_sec=5):
    start = time.time()
    try:
        result = subprocess.run(['python3', '-c', code], input=stdin_data,
                                 capture_output=True, text=True, timeout=timeout_sec)
        elapsed = time.time() - start
        return result.stdout.strip(), result.returncode == 0, elapsed
    except subprocess.TimeoutExpired:
        return None, False, timeout_sec

def check_functional_correctness(code, test_cases):
    for case in test_cases:
        output, success, _ = run_with_timeout(code, case['stdin'])
        if not success or output != case['expected']:
            return False, {**case, 'actual_output': output if output is not None else '(crashed or timed out)'}
    return True, None

def measure_runtime(code, test_cases, repeats=3):
    total_time = 0
    for _ in range(repeats):
        for case in test_cases:
            _, success, elapsed = run_with_timeout(code, case['stdin'])
            if not success:
                return None
            total_time += elapsed
    return total_time / repeats

def validate_repair(slow_code, repaired_code, problem_id, test_cases_dir, speedup_threshold=SPEEDUP_THRESHOLD):
    test_cases = load_test_cases(problem_id, test_cases_dir)
    if not test_cases:
        return {'verdict': 'RETRY', 'reason': f'No test cases found for {problem_id}'}
    compiles, syntax_error = check_compiles(repaired_code)
    if not compiles:
        return {'verdict': 'RETRY', 'reason': 'compile_error', 'detail': syntax_error}
    correct, failing_case = check_functional_correctness(repaired_code, test_cases)
    if not correct:
        return {'verdict': 'RETRY', 'reason': 'test_failure', 'detail': failing_case}
    slow_runtime = measure_runtime(slow_code, test_cases)
    repaired_runtime = measure_runtime(repaired_code, test_cases)
    if slow_runtime is None or repaired_runtime is None or repaired_runtime <= 0:
        return {'verdict': 'RETRY', 'reason': 'runtime_measurement_failed'}
    speedup = slow_runtime / repaired_runtime
    if speedup >= speedup_threshold:
        return {'verdict': 'ACCEPT', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}
    else:
        return {'verdict': 'RETRY', 'reason': 'insufficient_speedup', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}

def load_completed_problem_ids(output_path):
    completed = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                try:
                    completed.add(json.loads(line)['problem_id'])
                except Exception:
                    continue
    return completed

print('Validation + checkpoint helpers ready.')

In [ ]:
def build_all_patterns_summary(kb):
    sections = []
    for pattern_id in ['P1', 'P2', 'P3', 'P4', 'P5', 'P6']:
        entry = kb[pattern_id]
        poi = entry.get('poi', {})
        sections.append(f"""{pattern_id} - {entry.get('pattern_name', '')}
Transformation: {poi.get('transformation', '')}
Activation conditions: {', '.join(poi.get('activation_conditions', []))}
Constraints: {', '.join(poi.get('constraints', []))}
Expected benefit: {poi.get('expected_benefit', '')}""")
    return "\n\n".join(sections)


def build_baseline_prompt(all_patterns_summary, slow_code):
    return f"""You are an expert Python engineer specializing in performance optimization.

Here are six common optimization patterns and their guidance:

{all_patterns_summary}

Slow code:
{slow_code}

Identify which pattern(s), if any, apply to this code, and rewrite it to apply the
appropriate transformation. The optimized code must:
- Preserve the original function signature and behavior
- Pass all existing test cases
- Achieve a real performance improvement where possible

Return ONLY the optimized Python code. No markdown. No explanations."""


def generate_baseline_repair(all_patterns_summary, slow_code):
    prompt = build_baseline_prompt(all_patterns_summary, slow_code)
    response = call_gen_llm(prompt, max_new_tokens=800)
    return extract_code_from_response(response)


def build_baseline_feedback_prompt(all_patterns_summary, slow_code, previous_code, validation_result):
    reason = validation_result.get('reason')

    if reason == 'compile_error':
        feedback = f"Your previous attempt had a syntax error: {validation_result.get('detail')}"
    elif reason == 'test_failure':
        case = validation_result.get('detail', {})
        feedback = (
            f"Your previous attempt failed a test case.\n"
            f"Input:\n{case.get('stdin', '')}\n"
            f"Expected output:\n{case.get('expected', '')}\n"
            f"Your code's ACTUAL output:\n{case.get('actual_output', '')}\n"
            f"Compare carefully and fix the logic error."
        )
    elif reason == 'insufficient_speedup':
        feedback = (
            f"Your previous attempt was correct but only {validation_result.get('speedup')}x speedup "
            f"(need >= {SPEEDUP_THRESHOLD}x). Try a genuinely more efficient approach."
        )
    else:
        feedback = "Your previous attempt did not pass validation. Try a different approach."

    base_prompt = build_baseline_prompt(all_patterns_summary, slow_code)
    return f"""{base_prompt}

Your previous attempt:
{previous_code}

Feedback:
{feedback}

Generate a NEW, corrected version."""


def baseline_repair_with_feedback_loop(all_patterns_summary, slow_code, problem_id, test_cases_dir, max_attempts=MAX_ATTEMPTS):
    current_code = generate_baseline_repair(all_patterns_summary, slow_code)
    history = []
    last_actual_output = None

    for attempt in range(1, max_attempts + 1):
        result = validate_repair(slow_code, current_code, problem_id, test_cases_dir)
        result['attempt'] = attempt
        history.append(result)

        if result['verdict'] == 'ACCEPT':
            return {'final_verdict': 'ACCEPT', 'code': current_code, 'history': history}

        current_actual = result.get('detail', {}).get('actual_output') if isinstance(result.get('detail'), dict) else None
        if current_actual is not None and current_actual == last_actual_output:
            return {'final_verdict': 'RETRY_EXHAUSTED_STUCK', 'code': current_code, 'history': history}
        last_actual_output = current_actual

        if attempt == max_attempts:
            return {'final_verdict': 'RETRY_EXHAUSTED', 'code': current_code, 'history': history}

        feedback_prompt = build_baseline_feedback_prompt(all_patterns_summary, slow_code, current_code, result)
        response = call_gen_llm(feedback_prompt, max_new_tokens=800)
        current_code = extract_code_from_response(response)

    return {'final_verdict': 'RETRY_EXHAUSTED', 'code': current_code, 'history': history}

print('Single-LLM baseline logic ready.')

In [ ]:
def run_baseline_batch(test_samples, kb, test_cases_dir, output_path, max_attempts=MAX_ATTEMPTS):
    all_patterns_summary = build_all_patterns_summary(kb)
    completed = load_completed_problem_ids(output_path)
    print(f'Already completed: {len(completed)} / {len(test_samples)}')

    with open(output_path, 'a') as out_f:
        for i, sample in enumerate(test_samples):
            problem_id = sample.get('problem_id')
            slow_code = sample['input']

            if problem_id in completed:
                continue

            print(f'[{i+1}/{len(test_samples)}] Processing {problem_id}...')
            start_time = time.time()

            try:
                loop_result = baseline_repair_with_feedback_loop(
                    all_patterns_summary, slow_code, problem_id, test_cases_dir, max_attempts=max_attempts
                )
                elapsed = time.time() - start_time
                result_record = {
                    'problem_id': problem_id,
                    'final_verdict': loop_result['final_verdict'],
                    'num_attempts': len(loop_result['history']),
                    'history': loop_result['history'],
                    'final_code': loop_result['code'],
                    'elapsed_seconds': round(elapsed, 2),
                }
            except Exception as e:
                result_record = {'problem_id': problem_id, 'final_verdict': 'PIPELINE_ERROR', 'error': str(e)}

            out_f.write(json.dumps(result_record) + '\n')
            out_f.flush()

    print('\nBaseline batch complete.')


run_baseline_batch(test_samples, pattern_knowledge_base, TEST_CASES_DIR, BASELINE_OUTPUT_PATH, max_attempts=MAX_ATTEMPTS)

In [ ]:
import statistics

def score_results(path, label):
    results = []
    with open(path) as f:
        for line in f:
            results.append(json.loads(line))
    total = len(results)
    accepted = [r for r in results if r.get('final_verdict') == 'ACCEPT']
    pass1_count = sum(
        1 for r in results
        if r.get('history') and (
            r['history'][0].get('verdict') == 'ACCEPT' or
            r['history'][0].get('reason') == 'insufficient_speedup'
        )
    )
    print(f'=== {label} ===')
    print(f'Total: {total}')
    print(f'%OPT: {100*len(accepted)/total:.1f}%')
    print(f'Pass@1: {100*pass1_count/total:.1f}%')
    speedups = [r['history'][-1]['speedup'] for r in accepted if 'speedup' in r['history'][-1]]
    if speedups:
        print(f'Median speedup: {statistics.median(speedups):.3f}x')
        print(f'Mean speedup: {statistics.mean(speedups):.3f}x')

score_results(BASELINE_OUTPUT_PATH, 'Single-LLM (GPT-5.4-mini)')